In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3
import random
import ast
from estnltk import Text
from estnltk_neural.taggers import StanzaSyntaxTagger
from conllu.serializer import serialize_field

### Setup

In [3]:
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
SOURCE_DATA = "verb_obj_cases_filtered.db"
RESULT_DATA_PATH = f"{SOURCE_DATA_PATH}obj_conflict_dataset/"
RESULT_OLEMA = "verb_olema.conll"
RESULT_ONE_DEPENDANT = "verb_has_one_dependant.conll"
RESULT_NOM_NOT_IMPER = "obj_nom_not_imperative.conll"
RESULT_WRONG_POS = "obj_POS_wrong.conll"

In [4]:
stanza_tagger = StanzaSyntaxTagger()

### Methods

In [6]:
def get_table_data(db_path: str, table_name: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT sentence_id, form_span, sentence FROM {table_name}".format(table_name=table_name))
    result = cursor.fetchall()
    conn.close()
    return result

In [7]:
def get_table_data_verb_compound(db_path: str, table_name: str, verb: str, verb_compound: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT sentence_id, form_span, sentence FROM {table_name} WHERE verb='{verb}' AND verb_compound='{verb_compound}'".format(table_name=table_name, verb=verb, verb_compound=verb_compound))
    result = cursor.fetchall()
    conn.close()
    return result

In [8]:
def get_distinct_verb_compound(db_path: str, table_name: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT verb, verb_compound FROM {table_name}".format(table_name=table_name))
    result = cursor.fetchall()
    conn.close()
    return result

In [9]:
def layer_to_custom_conll(
        text,
        layer,
        sentence_id,
        target_span,
        target_value,
        sentences_layer="sentences"
    ):

    CONLL_ATTRIBUTES = [
        "id",
        "form",
        "upostag",
        "feats",
        "head",
        "deprel"
    ]

    convertable_layer = text[layer]
    layer_has_form = "form" in convertable_layer.attributes

    output = []

    # Sentence ID line
    output.append(f"# sent_id = {sentence_id}")
    # Sentence text line
    output.append(f"# text = {text.text}")

    # There is exactly one sentence
    sentence = text[sentences_layer][0]

    for word in sentence:

        annotation = convertable_layer.get(
            word.base_span
        ).annotations[0]

        values = []

        for attr in CONLL_ATTRIBUTES:

            if attr == "form" and not layer_has_form:
                value = word.text
            else:
                value = annotation[attr]

            value = serialize_field(value)
            values.append(value)

        # Error column
        if (word.start, word.end) == target_span:
            error = target_value
        else:
            error = "_"

        values.append(serialize_field(error))

        output.append("\t".join(values))

    # Empty line after sentence
    output.append("")

    return "\n".join(output)

In [10]:
def n_random_examples_to_conll(table_rows: list, save_path: str, n_examples: int):
    random.seed(0)
    random.shuffle(table_rows)

    table_rows_n = table_rows[:n_examples]

    for idx, row in enumerate(table_rows_n):
        sentence_text = Text(row[2]).tag_layer()
        stanza_tagger.tag(sentence_text)
        target_span = ast.literal_eval(row[1])
        conll_string = layer_to_custom_conll(
            text=sentence_text,
            layer="stanza_syntax",
            sentence_id=row[0],
            target_span=target_span,
            target_value="CHECK",
            sentences_layer="sentences"
        )
        with open(save_path, "a", encoding="UTF-8") as f:
            f.write(conll_string)
            f.write("\n")

### I *Olema*-verb constructions

In [11]:
verb_compounds = get_distinct_verb_compound(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "verb_olema")

In [12]:
len(verb_compounds)

30

In [15]:
table_rows_olema = get_table_data(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "verb_olema")

In [16]:
len(table_rows_olema)

147

In [17]:
n_random_examples_to_conll(table_rows_olema, f"{RESULT_DATA_PATH}{RESULT_OLEMA}", 20)

### II Other verb constructions from list of verbs that cannot have obj

In [18]:
other_verb_compounds = get_distinct_verb_compound(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "verbs_without_obj")

In [19]:
len(other_verb_compounds)

93

In [20]:
for compound in other_verb_compounds:
    table_rows_other = get_table_data_verb_compound(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "verbs_without_obj", compound[0], compound[1])
    n_random_examples_to_conll(table_rows_other, f"{RESULT_DATA_PATH}verb_{compound[0]}_{compound[1]}.conll", 20)

### III Verb has only one main dependant in transactions database

In [21]:
table_rows_one_dep = get_table_data(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "verbs_one_dependant")

In [22]:
len(table_rows_one_dep)

6877

In [23]:
n_random_examples_to_conll(table_rows_one_dep, f"{RESULT_DATA_PATH}{RESULT_ONE_DEPENDANT}", 20)

### IV Obj case singular nom, but verb mood isn't imperative

In [24]:
table_rows_sg_nom = get_table_data(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "nom_not_imper")

In [25]:
len(table_rows_sg_nom)

12842

In [26]:
n_random_examples_to_conll(table_rows_sg_nom, f"{RESULT_DATA_PATH}{RESULT_NOM_NOT_IMPER}", 20)

### V Obj part-of-speech is incompatible

In [27]:
table_rows_pos = get_table_data(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "wrong_POS")

In [28]:
len(table_rows_pos)

1248

In [29]:
n_random_examples_to_conll(table_rows_pos, f"{RESULT_DATA_PATH}{RESULT_WRONG_POS}", 20)